# 🔘🖼️ Gradio SDXL Turbo Text-to-Image

⚠️ **Remember to copy this notebook in your Drive and rename.**

*Workflows for IAAC MaCDA GenAI  (Apr - Jun 2026) taught by [James McBennett](https://www.linkedin.com/in/mcbennett/) and [Aymeric Brouez](https://www.linkedin.com/in/aymeric-brouez/)*

*With special thanks to past faculty [Nono Martínez Alonso](https://youtube.com/NonoMartinezAlonso).*

**🗒️ Gradio's [Documentation](https://www.gradio.app/docs)**

Gradio's [Documentation](https://www.gradio.app/docs)

## Install Gradio (and Restart)

In [ ]:
!pip install gradio --quiet

### Mount Drive

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## Setup

In [ ]:
%cd /content
!rm -rf iaac_genai
!git clone https://github.com/jamesmcbennett/iaac_genai
%cd /content/iaac_genai/

In [ ]:
import sys
sys.path.append('/content/iaac_genai')

In [ ]:
!pip install -q -r requirements.txt --quiet > /dev/null 2>&1

In [ ]:
from config import Config
from utils import set_image_path, save_image, save_yml, save_svg
import torch

from diffusers.utils import load_image
from PIL import Image, ImageDraw, ImageFont
import gradio as gr
import numpy as np
import random
import os
from datetime import datetime

## Ouput Directory

In [ ]:
Config.OUTPUT_DIR = "/content/drive/MyDrive/output"
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)

## Load SDXL Turbo Pipeline

In [ ]:
from diffusers import AutoPipelineForText2Image
pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16).to(Config.TORCH_DEVICE)

## Function

In [ ]:
def infer(prompt, negative_prompt, seed, randomize_seed, width, height, guidance_scale, num_inference_steps, progress=gr.Progress(track_tqdm=True)):

    if randomize_seed:
        seed = random.randint(0, MAX_SEED)

    generator = torch.Generator().manual_seed(seed)
    image = pipe(prompt=prompt, negative_prompt=negative_prompt, guidance_scale=guidance_scale, num_inference_steps=num_inference_steps, width=width, height=height, generator=generator).images[0]

    Config.PROMPT = prompt
    Config.SEED = seed
    Config.STEPS = num_inference_steps

    Config.AUTHOR = "James"
    Config.ALGO_TYPE = 'Gradio Text to Image'
    Config.ALGO_NAME = 'SDXL Turbo'

    set_image_path()
    save_image(image)

    return image, seed

## Run Gradio

In [ ]:
examples = [
    "Astronaut in a jungle, cold color palette, muted colors, detailed, 8k",
    "An astronaut riding a green horse",
    "A delicious ceviche cheesecake slice",
    "A fox wearing a suit reading a newspaper in a cafe",
    "Neon cyberpunk street market at night, rain reflections",
    "A polar bear surfing a massive wave",
    "Ancient Roman soldier using a smartphone",
    "A treehouse library at sunset, warm light",
    "Dragon sleeping on a pile of gold coins, cinematic",
    "A cat conducting an orchestra, concert hall",
    "Underwater city with glowing jellyfish, bioluminescent",
    "Viking longship in space, stars, dramatic lighting",
    "A robot painting a self portrait, studio, oil on canvas",
    "Giant mushrooms in a misty forest, fantasy",
    "A samurai standing in a field of cherry blossoms",
    "Steampunk hot air balloon over Victorian London",
    "A lighthouse on a cliff during a thunderstorm",
    "Wolf howling at a double moon, alien landscape",
    "A medieval knight at a drive-through restaurant",
    "Zen garden on the surface of Mars, astronaut meditating",
]

MAX_SEED = 2048
MAX_IMAGE_SIZE = 1024

css = """
#col-container {
    margin: 0 auto;
    max-width: 640px;
}
"""

with gr.Blocks(css=css) as demo:
    with gr.Column(elem_id="col-container"):
        gr.Markdown(" # Text-to-Image Gradio SDXL Turbo")

        with gr.Row():
            prompt = gr.Text(
                label="Prompt",
                show_label=False,
                max_lines=1,
                placeholder="Enter your prompt",
                container=False,
            )

            run_button = gr.Button("Run", scale=0, variant="primary")

        result = gr.Image(label="Result", show_label=False)

        with gr.Accordion("Advanced Settings", open=False):
            negative_prompt = gr.Text(
                label="Negative prompt",
                max_lines=1,
                placeholder="Enter a negative prompt",
                visible=False,
            )

            seed = gr.Slider(
                label="Seed",
                minimum=0,
                maximum=MAX_SEED,
                step=1,
                value=0,
            )

            randomize_seed = gr.Checkbox(label="Randomize seed", value=True)

            with gr.Row():
                width = gr.Slider(
                    label="Width",
                    minimum=256,
                    maximum=MAX_IMAGE_SIZE,
                    step=32,
                    value=1024,
                )

                height = gr.Slider(
                    label="Height",
                    minimum=256,
                    maximum=MAX_IMAGE_SIZE,
                    step=32,
                    value=1024,
                )

            with gr.Row():
                guidance_scale = gr.Slider(
                    label="Guidance scale",
                    minimum=0.0,
                    maximum=10.0,
                    step=0.1,
                    value=0.0,
                )

                num_inference_steps = gr.Slider(
                    label="Number of inference steps",
                    minimum=1,
                    maximum=50,
                    step=1,
                    value=2,
                )

        gr.Examples(examples=examples, inputs=[prompt])

    gr.on(
        triggers=[run_button.click, prompt.submit],
        fn=infer,
        inputs=[
            prompt,
            negative_prompt,
            seed,
            randomize_seed,
            width,
            height,
            guidance_scale,
            num_inference_steps,
        ],
        outputs=[result, seed],
    )

demo.launch(share=True, debug=True)